In [ ]:
import pandas as pd
import re
import joblib
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [9]:
df = pd.read_csv("IMDB Dataset.csv")   
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df.head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [11]:
print(df['sentiment'].value_counts())
print(df['review'].apply(lambda x: len(x.split())).describe())

positive    25000
negative    25000
Name: sentiment, dtype: int64
count    50000.000000
mean       231.156940
std        171.343997
min          4.000000
25%        126.000000
50%        173.000000
75%        280.000000
max       2470.000000
Name: review, dtype: float64


In [12]:
def clean_text(text):
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_review'] = df['review'].apply(clean_text)


In [16]:
stop_words = set(stopwords.words('english')) - {'not', 'no', 'never', 'nor'}
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(words)

df['clean_review'] = df['clean_review'].apply(preprocess)



vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=15000,
    min_df=5,
    stop_words='english'
)
X = vectorizer.fit_transform(df['clean_review'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



In [17]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_model.predict(X_test)))

lr_model = LogisticRegression(max_iter=1000, C=1.0)
lr_model.fit(X_train, y_train)
print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_model.predict(X_test)))

svm_model = LinearSVC(C=1.0)
svm_model.fit(X_train, y_train)
print("SVM Accuracy:", accuracy_score(y_test, svm_model.predict(X_test)))


Naive Bayes Accuracy: 0.864
Logistic Regression Accuracy: 0.8944
SVM Accuracy: 0.8913


In [18]:
y_pred = lr_model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.90      0.88      0.89      5000
           1       0.89      0.91      0.90      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

[[4413  587]
 [ 469 4531]]


In [19]:
params = {'C': [0.1, 1, 10]}
grid = GridSearchCV(LogisticRegression(max_iter=1000), params, cv=5, scoring='f1')
grid.fit(X_train, y_train)
print("Best Params:", grid.best_params_)

best_model = grid.best_estimator_
best_model.fit(X_train, y_train)
print("Final Accuracy:", accuracy_score(y_test, best_model.predict(X_test)))

joblib.dump(best_model, "sentiment_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
print("Model saved successfully.")

Best Params: {'C': 1}
Final Accuracy: 0.8944
Model saved successfully.
